In [0]:
# Imports

from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
# Preparação dos endereços das tabelas

catalog = "atividade_2"

bronze = "bronze"
silver = "silver"
gold = "gold"

bronze_schema = f"{catalog}.{bronze}"
silver_schema = f"{catalog}.{silver}"
gold_schema = f"{catalog}.{gold}"

landing_path = f"/Volumes/{catalog}/{bronze}/Landing"

print(catalog)
print(bronze_schema)
print(silver_schema)
print(gold_schema)
print(landing_path)

## Dim Movies

In [0]:
df_info_silver = spark.table(f"{silver_schema}.tb_info_filmes")

df_dim_movies = (
    df_info_silver
        .select(
            F.monotonically_increasing_id().cast('BIGINT').alias("sk_movie_id"),
            F.col("id_filme").cast("string"),
            "titulo",
            "data_lancamento",
            "ano_lancamento",
            "duracao_minutos",
            "idioma_original",
            "status_filme",
            "sinopse"
        )
)

df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_movies")

print(f"dim_movies: {df_dim_movies.count()} linhas")

display(df_dim_movies.limit(5))

## Fact Movies Performance

In [0]:
df_engajamento_silver = spark.table(f"{silver_schema}.tb_metricas_engajamento")
df_financeiro_silver = spark.table(f"{silver_schema}.tb_financeiro_filmes")

df_dim_movies_lk = spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "id_filme")

df_fact_movies_performance = (
    df_dim_movies_lk
        .join(df_engajamento_silver, on="id_filme", how="left")
        .join(df_financeiro_silver, on="id_filme", how="left")
        .select(
            "sk_movie_id",
            "orcamento_usd",
            "receita_usd",
            "lucro_usd",
            "orcamento_brl",
            "receita_brl",
            "lucro_brl",
            "popularidade",
            "nota_media_tmdb",
            "qtd_votos_tmdb",
            "nota_media_imdb",
            "qtd_votos_imdb"
        )
)

df_fact_movies_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fact_movies_performance")

print(f"fact_movies_performance: {df_fact_movies_performance.count()} linhas")

display(df_fact_movies_performance.limit(5))

## Dim Genres e Bridge Movie Genre

In [0]:
df_generos_silver = spark.table(f"{silver_schema}.tb_generos")

df_dim_genres = (
    df_generos_silver
        .dropDuplicates(["genero"])
        .select(
            F.monotonically_increasing_id().cast('BIGINT').alias("sk_genre_id"), 
            F.col("genero").alias("nome_genero")
        )
)

df_bridge_movie_genre = (
    df_generos_silver
        .select(
            "id_filme", 
            F.col("genero").alias("nome_genero")
        )
        .withColumn("id_filme", F.col("id_filme").cast("string"))
        .join(df_dim_movies_lk, on="id_filme", how="inner")
        .join(df_dim_genres, on="nome_genero", how="inner")
        .select("sk_genre_id", "sk_movie_id")
)

In [0]:
df_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_genres")

print(f"dim_genres: {df_dim_genres.count()} linhas")

display(df_dim_genres.limit(5))

In [0]:
df_bridge_movie_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_genre")

print(f"bridge_movie_genre: {df_bridge_movie_genre.count()} linhas")

display(df_bridge_movie_genre.limit(5))

## Dim People, Dim Companies e Bridges com Movie

In [0]:
df_pessoas_empresas_silver = spark.table(f"{silver_schema}.tb_pessoas_empresas")

df_dim_people = (
    df_pessoas_empresas_silver
        .where(F.col("tipo_entidade").isin(["Ator", "Diretor", "Roteirista"]))
        .dropDuplicates(["entidade", "tipo_entidade"])
        .select(
            F.monotonically_increasing_id().cast("BIGINT").alias("sk_person_id"),
            F.col("entidade").alias("nome_pessoa"),
            F.col("tipo_entidade").alias("tipo_pessoa")
        )
)

df_dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_people")

print(f"dim_people: {df_dim_people.count()} linhas")

display(df_dim_people.limit(5))

In [0]:
df_dim_companies = (
    df_pessoas_empresas_silver
        .where(F.col("tipo_entidade") == "Produtora")
        .dropDuplicates(["entidade"])
        .select(
            F.monotonically_increasing_id().cast("BIGINT").alias("sk_company_id"),
            F.col("entidade").alias("nome_produtora"),
        )
)

df_dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_companies")

print(f"dim_companies: {df_dim_companies.count()} linhas")

display(df_dim_companies.limit(5))

In [0]:
df_dim_people_lk = spark.table(f"{gold_schema}.dim_people")

df_bridge_movie_person = (
    df_pessoas_empresas_silver
        .join(df_dim_movies_lk, on="id_filme", how="inner")
        .withColumns({
            "nome_pessoa": F.col("entidade"),
            "tipo_pessoa": F.col("tipo_entidade")
        })
        .join(df_dim_people_lk, on=["nome_pessoa", "tipo_pessoa"], how="inner")
        .select(
            F.col("sk_movie_id"),
            F.col("sk_person_id")
        )
)

df_bridge_movie_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_person")

print(f"bridge_movie_person: {df_bridge_movie_person.count()} linhas")

display(df_bridge_movie_person.limit(5))

In [0]:
df_dim_companies_lk = spark.table(f"{gold_schema}.dim_companies")

df_bridge_movie_company = (
    df_pessoas_empresas_silver
        .join(df_dim_movies_lk, on="id_filme", how="inner")
        .withColumns({
            "nome_produtora": F.col("entidade"),
            "tipo_pessoa": F.col("tipo_entidade")
        })
        .join(df_dim_companies_lk, on=["nome_produtora"], how="inner")
        .select(
            F.col("sk_movie_id"),
            F.col("sk_company_id")
        )
)

df_bridge_movie_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_company")

print(f"bridge_movie_company: {df_bridge_movie_company.count()} linhas")

display(df_bridge_movie_company.limit(5))

## Dim reviews

In [0]:
df_avaliacoes_usuarios_silver = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")

df_dim_reviews = (
    df_avaliacoes_usuarios_silver
        .groupBy("id_filme")
        .agg(
            F.count("id_filme").alias("qtd_avaliacoes_usuarios"),
            F.round(F.avg(F.col("nota_usuario")), 2).alias("nota_media_usuarios")
        ).withColumn("sk_review_id", F.monotonically_increasing_id().cast('BIGINT'))
        .join(df_dim_movies_lk, how="inner", on="id_filme")
        .select(
            F.col("sk_review_id"),
            F.col("sk_movie_id"),
            F.col("qtd_avaliacoes_usuarios"),
            F.col("nota_media_usuarios")
        )
)

df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_reviews")

print(f"dim_reviews: {df_dim_reviews.count()} linhas")

display(df_dim_reviews.limit(5))